In [1]:
from kfp import dsl
from kfp.dsl import (
    Input, Output, Dataset, Model, component, pipeline
)
from google.cloud import aiplatform
from components.components import vertex_hyperparameter_tuning_component,custom_job_training_component,upload_model_component,vertex_batch_predict_bigquery_component
from typing import Any, Dict, List, Optional, Text, Tuple, Union, Sequence, Callable
import yaml
from kfp.v2 import compiler

In [11]:
# Cell 1: Load config
config_path = "config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)  
    constants = config.get("runtime", {})
    model_cfg = config.get("model_upload", {})

In [12]:
model_cfg

{'model_gcs_location': 'gs://hcm-cm-de-code-hcb-dev/vertex-test/model',
 'serving_container_image_uri': 'us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-6:latest',
 'model_display_name': 'pre_term_maternity_model',
 'upload_to_existing_model': True,
 'existing_model_resource_name': 'projects/anbc-dev-hcm-cm-de/locations/us-east4/models/1727284388724473856',
 'description': 'Model uploaded via pipeline'}

# Model Upload

In [4]:
# Cell 2: Define pipeline
@dsl.pipeline(
    name="model-upload-pipeline",
    description="Simple pipeline to upload model to Vertex AI Model Registry"
)
def model_upload_pipeline(
    project: str,
    service_account: str,
    cmek_key: str,
    location: str,
    model_gcs_location: str,
    serving_container_image_uri: str,
    model_display_name: str,
    upload_to_existing_model: bool = False,
    existing_model_resource_name: str = "",
    description: str = None,
    serving_container_predict_route: str = "/predict",
    serving_container_health_route: str = "/health",
    model_framework: str = "XGBoost",
    model_type: str = "classifier",
):
    # Upload model component
    model_register = upload_model_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        model_gcs_location=model_gcs_location,
        serving_container_image_uri=serving_container_image_uri,
        model_display_name=model_display_name,
        upload_to_existing_model=upload_to_existing_model,
        existing_model_resource_name=existing_model_resource_name,
        description=description,
        serving_container_predict_route=serving_container_predict_route,
        serving_container_health_route=serving_container_health_route,
        model_framework=model_framework,
        model_type=model_type,
    )

In [5]:
# Cell 3: Compile pipeline
compiler.Compiler().compile(
    pipeline_func=model_upload_pipeline,
    package_path="model_upload_pipeline.yaml"
)
print("Pipeline compiled to model_upload_pipeline.yaml")

Pipeline compiled to model_upload_pipeline.yaml


In [6]:
# Cell 4: Run pipeline
parameter_values = {
    "project": constants["project"],
    "service_account": constants["service_account"],
    "cmek_key": constants["cmek_key"],
    "location": constants["location"],
    "model_gcs_location": model_cfg.get("model_gcs_location"),
    "serving_container_image_uri": model_cfg.get("serving_container_image_uri"),
    "model_display_name": model_cfg.get("model_display_name"),
    "upload_to_existing_model": model_cfg.get("upload_to_existing_model", False),
    "existing_model_resource_name": model_cfg.get("existing_model_resource_name", ""),
    "description": model_cfg.get("description"),
    "serving_container_predict_route": model_cfg.get("serving_container_predict_route", "/predict"),
    "serving_container_health_route": model_cfg.get("serving_container_health_route", "/health"),
    "model_framework": model_cfg.get("model_framework", "XGBoost"),
    "model_type": model_cfg.get("model_type", "classifier"),
}

aiplatform.init(
    project=constants["project"], 
    location=constants["location"],
    service_account=constants["service_account"]
)

pipeline_job = aiplatform.PipelineJob(
    display_name="model-upload-pipeline",
    template_path="model_upload_pipeline.yaml",
    pipeline_root=constants["pipeline_root"],
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)

pipeline_job.run(sync=True)

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/model-upload-pipeline-20260102133342?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342 current state:
3
PipelineJob run completed. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/model-upload-pipeline-20260102133342


# Batch Preds

In [4]:
# Cell 7: Load batch prediction config
config_path = "config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)  
    constants = config.get("runtime", {})
    batch_cfg = config.get("batch_prediction", {})

In [5]:
batch_cfg

{'model_resource_name': 'projects/anbc-dev-hcm-cm-de/locations/us-east4/models/1727284388724473856@2',
 'key_field': 'asdb_member_key',
 'input_table': 'anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_selected_features_test_01_23_2026_19_52_43',
 'output_table': 'anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_selected_features_test_01_23_2026_19_52_43_prediction',
 'compute_dataset': 'hcm_cm_de_beam_dev_hcm_cm_de',
 'expiration_days': 2,
 'excluded_fields': ['pre_term_max'],
 'included_fields': [],
 'machine_type': {'machine_type': 'n1-standard-16'},
 'starting_replica_count': 1,
 'max_replica_count': 1}

In [6]:
# Cell 8: Define batch prediction pipeline
@dsl.pipeline(
    name="batch-prediction-pipeline",
    description="Pipeline for Vertex AI batch prediction with BigQuery input/output"
)
def batch_prediction_pipeline(
    project: str,
    location: str,
    service_account: str,
    cmek_key: str,
    cost_center: str,
    owner: str,
    # Model details
    model_resource_name: str,
    # BigQuery specific
    key_field: str,
    input_table: str,
    output_table: str,
    compute_dataset: str,
    expiration_days: int = 30,
    # Instance configuration - field filtering
    excluded_fields: list = None,
    included_fields: list = None,
    # Machine configuration
    machine_type: dict = None,
    # Job configuration
    starting_replica_count: int = 1,
    max_replica_count: int = 1,
    batch_size: int = None,
):
    # Batch prediction component
    batch_pred_job = vertex_batch_predict_bigquery_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        cost_center=cost_center,
        owner=owner,
        model_resource_name=model_resource_name,
        key_field=key_field,
        input_table=input_table,
        output_table=output_table,
        compute_dataset=compute_dataset,
        expiration_days=expiration_days,
        excluded_fields=excluded_fields,
        included_fields=included_fields,
        machine_type=machine_type or {"machine_type": "n1-standard-16"},
        starting_replica_count=starting_replica_count,
        max_replica_count=max_replica_count,
        batch_size=batch_size,
    )

In [7]:
# Cell 9: Compile batch prediction pipeline
compiler.Compiler().compile(
    pipeline_func=batch_prediction_pipeline,
    package_path="batch_prediction_pipeline.yaml"
)
print("Batch prediction pipeline compiled to batch_prediction_pipeline.yaml")

Batch prediction pipeline compiled to batch_prediction_pipeline.yaml


In [8]:
# Cell 10: Run batch prediction pipeline
parameter_values = {
    "project": constants["project"],
    "service_account": constants["service_account"],
    "cmek_key": constants["cmek_key"],
    "location": constants["location"],
    "cost_center": constants["costcenter"],
    "owner": constants["owner"],
    "model_resource_name": batch_cfg.get("model_resource_name"),
    "key_field": batch_cfg.get("key_field"),
    "input_table": batch_cfg.get("input_table"),
    "output_table": batch_cfg.get("output_table"),
    "compute_dataset": batch_cfg.get("compute_dataset"),
    "expiration_days": batch_cfg.get("expiration_days", 30),
    "excluded_fields": batch_cfg.get("excluded_fields"),
    "included_fields": batch_cfg.get("included_fields"),
    "machine_type": batch_cfg.get("machine_type", {"machine_type": "n1-standard-16"}),
    "starting_replica_count": batch_cfg.get("starting_replica_count", 1),
    "max_replica_count": batch_cfg.get("max_replica_count", 1),
    "batch_size": batch_cfg.get("batch_size"),
}

aiplatform.init(
    project=constants["project"], 
    location=constants["location"],
    service_account=constants["service_account"]
)

pipeline_job = aiplatform.PipelineJob(
    display_name="batch-prediction-pipeline",
    template_path="batch_prediction_pipeline.yaml",
    pipeline_root=constants["pipeline_root"],
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)

pipeline_job.run(sync=True)

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/batch-prediction-pipeline-20260124184635?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/batch-prediction-pipeline-20260124184635 current state:
3
PipelineJob proje

RuntimeError: Job failed with:
code: 9
message: " The DAG failed because some tasks failed. The failed tasks are: [vertex-batch-predict-bigquery-component].; Job (project_id = anbc-dev-hcm-cm-de, job_id = 3939740377833013248) is failed due to the above error.; Failed to handle the job: {project_number = 46378383599, job_id = 3939740377833013248}"


In [ ]:
import pandas as pd
from google.cloud import bigquery

project = "anbc-hcb-dev"
table2 = "cm_medicaid_hcb_dev.a974930_prod_data_maternity_all_predictors"

client = bigquery.Client(project=project)

df2 = client.list_rows(table2).to_dataframe(create_bqstorage_client=True)

print(df2.shape)

(31653, 618)


In [32]:
df2 = df2[['asdb_member_key', 'index_dt','mom_age', 'multi', 'bleeding_in_current_preg', 'Cocaine',
       'Nicotine', 'lab_hCG', 'lab_hba1c_current', 'lab_glucose_pre',
       'glucose_challenge_pre', 'lab_hdl', 'lab_ldl', 'lab_triglyc_pre',
       'lab_triglyc_current', 'lab_creat', 'lab_altsgpt', 'lab_bilirub',
       'lab_sodium', 'lab_ferritin', 'sum_ed_visits_yr1',
       'low_sev_ed_visits_yr1', 'med_high_sev_ed_visits_yr1',
       'emis_ed_clm_yr1', 'emis_ed_clm_yr2', 'emis_ip_clm_yr2',
       'emis_lab_clm_yr2', 'coe_ip_hos_clm_yr2', 'coe_lab_clm_yr2',
       'coe_op_hos_clm_yr2', 'coe_maternity_clm_yr2', 'coe_mh_clm_yr2',
       'coe_surg_clm_yr2', 'abdominal_pain', 'CHO', 'VNA', 'MOH', 'HYP',
       'immune', 'bipolar', 'major_chronic_cnt', 'rx_claim_cnt_yr1',
       'gpi4_cnt_yr1', 'retail_fills_yr1', 'ss_brand_fills_yr1',
       'formulary_fills_yr1', 'maint_drug_fills_yr1',
       'antidiabetic_days_supply_yr1', 'beta_blocker_days_supply_yr1',
       'antidepressant_scripts_yr1', 'antidepressant_days_supply_yr1',
       'days_supply_sum_yr2', 'ss_brand_fills_yr2',
       'beta_blocker_days_supply_yr2', 'calcium_channel_blk_days_supply_yr2',
       'antidepressant_days_supply_yr2', 'tenure_yr1', 'tenure_yr2',
       'zip_weight_avg_medinc', 'acs_social_risk_score', 'sdi_score',
       'svi_score', 'adi_score', 'citizenship_index', 'education_index',
       'food_access', 'health_access', 'health_habits', 'housing_desert',
       'housing_ownership', 'housing_quality', 'income_index',
       'income_inequality', 'language_score', 'natural_disaster',
       'poverty_score', 'proactive_health', 'racial_diversity',
       'social_isolation', 'technology_access', 'transport_access',
       'unemployment_index', 'water_quality', 'disability_score',
       'health_infra', 'csdi_social_risk_score', 'sum_spec', 'cms_sti_scrn',
       'emb31', 'emb39', 'emb49', 'emb90', 'emb131', 'emb157', 'emb177',
       'emb181', 'emb203', 'urbsubr', 'pre_term_max']]

In [33]:
df2 = df2.copy()
df2.loc[:, 'urbsubr_S'] = (df2['urbsubr'] == 'S').astype(int)
df2.loc[:, 'gest_age'] = 0

In [34]:
df2 = df2[[ 'gest_age', 'mom_age', 'multi', 'bleeding_in_current_preg', 'Cocaine',
       'Nicotine', 'lab_hCG', 'lab_hba1c_current', 'lab_glucose_pre',
       'glucose_challenge_pre', 'lab_hdl', 'lab_ldl', 'lab_triglyc_pre',
       'lab_triglyc_current', 'lab_creat', 'lab_altsgpt', 'lab_bilirub',
       'lab_sodium', 'lab_ferritin', 'sum_ed_visits_yr1',
       'low_sev_ed_visits_yr1', 'med_high_sev_ed_visits_yr1',
       'emis_ed_clm_yr1', 'emis_ed_clm_yr2', 'emis_ip_clm_yr2',
       'emis_lab_clm_yr2', 'coe_ip_hos_clm_yr2', 'coe_lab_clm_yr2',
       'coe_op_hos_clm_yr2', 'coe_maternity_clm_yr2', 'coe_mh_clm_yr2',
       'coe_surg_clm_yr2', 'abdominal_pain', 'CHO', 'VNA', 'MOH', 'HYP',
       'immune', 'bipolar', 'major_chronic_cnt', 'rx_claim_cnt_yr1',
       'gpi4_cnt_yr1', 'retail_fills_yr1', 'ss_brand_fills_yr1',
       'formulary_fills_yr1', 'maint_drug_fills_yr1',
       'antidiabetic_days_supply_yr1', 'beta_blocker_days_supply_yr1',
       'antidepressant_scripts_yr1', 'antidepressant_days_supply_yr1',
       'days_supply_sum_yr2', 'ss_brand_fills_yr2',
       'beta_blocker_days_supply_yr2', 'calcium_channel_blk_days_supply_yr2',
       'antidepressant_days_supply_yr2', 'tenure_yr1', 'tenure_yr2',
       'zip_weight_avg_medinc', 'acs_social_risk_score', 'sdi_score',
       'svi_score', 'adi_score', 'citizenship_index', 'education_index',
       'food_access', 'health_access', 'health_habits', 'housing_desert',
       'housing_ownership', 'housing_quality', 'income_index',
       'income_inequality', 'language_score', 'natural_disaster',
       'poverty_score', 'proactive_health', 'racial_diversity',
       'social_isolation', 'technology_access', 'transport_access',
       'unemployment_index', 'water_quality', 'disability_score',
       'health_infra', 'csdi_social_risk_score', 'sum_spec', 'cms_sti_scrn',
       'emb31', 'emb39', 'emb49', 'emb90', 'emb131', 'emb157', 'emb177',
       'emb181', 'emb203', 'urbsubr_S', 'pre_term_max', 'asdb_member_key', 'index_dt']]

In [35]:
len(df2.columns)

100